# Notebook 04: Extension to Women's ODI & T20 Formats

Re-runs the full pipeline (DLS + ML) for:
- Women's ODI
- Men's T20I
- Women's T20I

Plus cross-format comparison and transfer learning experiment.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

from src.data_collection import load_processed
from src.data_processing import (
    filter_completed_matches, create_over_snapshots,
    add_match_context, time_based_train_test_split, PROCESSED_DIR
)
from src.dls_method import DLSModel
from src.feature_engineering import add_dls_features, prepare_features
from src.ml_models import train_all_models, predict_with_model
from src.evaluation import (
    full_evaluation, compute_metrics, cross_format_heatmap_data, generate_latex_table
)
from src.visualizations import plot_cross_format_heatmap, plot_model_comparison_bar

## 4.1 Define Pipeline Function

In [ ]:
def run_full_pipeline(format_key, overs_limit, xgb_trials=50, rf_trials=30, lgb_trials=30, nn_trials=15):
    """
    Run complete pipeline for a format:
    1. Load data
    2. Filter and create snapshots
    3. Fit DLS
    4. Train ML models
    5. Evaluate
    """
    print(f"\n{'='*60}")
    print(f"Processing: {format_key} (overs_limit={overs_limit})")
    print(f"{'='*60}")
    
    # Load and filter
    matches_df, deliveries_df = load_processed(format_key)
    matches_df, deliveries_df = filter_completed_matches(matches_df, deliveries_df)
    
    # Create snapshots
    snapshots = create_over_snapshots(deliveries_df, matches_df, overs_limit=overs_limit)
    snapshots = add_match_context(snapshots, matches_df)
    
    # Fit DLS
    dls = DLSModel(overs_limit=overs_limit)
    dls.fit(snapshots)
    dls.save()
    
    # Add DLS features
    snapshots = add_dls_features(snapshots, dls)
    
    # Train/test split
    train_df, test_df = time_based_train_test_split(snapshots, matches_df)
    
    # Save
    snapshots.to_parquet(PROCESSED_DIR / f'{format_key}_snapshots.parquet', index=False)
    train_df.to_parquet(PROCESSED_DIR / f'{format_key}_train.parquet', index=False)
    test_df.to_parquet(PROCESSED_DIR / f'{format_key}_test.parquet', index=False)
    
    # Prepare features
    X_train, y_train, feature_names = prepare_features(train_df)
    X_test, y_test, _ = prepare_features(test_df)
    
    # Train ML models
    model_results = train_all_models(
        X_train, y_train, X_test, y_test,
        format_key=format_key,
        xgb_trials=xgb_trials,
        rf_trials=rf_trials,
        lgb_trials=lgb_trials,
        nn_trials=nn_trials,
    )
    
    # Predictions
    predictions = {}
    if 'dls_predicted_final' in test_df.columns:
        predictions['DLS'] = test_df['dls_predicted_final'].fillna(y_test.mean()).values
    
    for name, data in model_results.items():
        predictions[name] = predict_with_model(data['model'], X_test, scaler=data.get('scaler'))
    
    # Evaluate
    eval_results = full_evaluation(test_df, predictions, format_key=format_key, overs_limit=overs_limit)
    
    # Visualizations
    plot_model_comparison_bar(eval_results['overall'], format_key=format_key)
    
    return {
        'eval_results': eval_results,
        'model_results': model_results,
        'predictions': predictions,
        'test_df': test_df,
        'dls': dls,
    }

## 4.2 Women's ODI

In [ ]:
womens_odi_results = run_full_pipeline('womens_odi', overs_limit=50)
womens_odi_results['eval_results']['overall']

## 4.3 Men's T20I

In [ ]:
mens_t20i_results = run_full_pipeline('mens_t20i', overs_limit=20)
mens_t20i_results['eval_results']['overall']

## 4.4 Women's T20I

In [ ]:
womens_t20i_results = run_full_pipeline('womens_t20i', overs_limit=20)
womens_t20i_results['eval_results']['overall']

## 4.5 Cross-Format Comparison

In [ ]:
# Load Men's ODI results too
from src.evaluation import TABLES_DIR
mens_odi_overall = pd.read_csv(TABLES_DIR / 'mens_odi_overall_comparison.csv')

# Cross-format heatmap data
format_results = {
    "Men's ODI": mens_odi_overall,
    "Women's ODI": womens_odi_results['eval_results']['overall'],
    "Men's T20I": mens_t20i_results['eval_results']['overall'],
    "Women's T20I": womens_t20i_results['eval_results']['overall'],
}

heatmap_df = cross_format_heatmap_data(format_results)
print("Cross-Format RMSE:")
print(heatmap_df)

# Save
heatmap_df.to_csv(TABLES_DIR / 'cross_format_rmse.csv')

In [ ]:
# Figure 7: Cross-format heatmap
plot_cross_format_heatmap(heatmap_df)
print("Saved cross-format heatmap")

## 4.6 Transfer Learning Experiment

Train on Men's ODI data, test on Women's ODI data.

In [ ]:
# Load Men's ODI training data and Women's ODI test data
from src.feature_engineering import load_prepared_data
from src.ml_models import load_model

# Men's ODI models
mens_odi_train = pd.read_parquet(PROCESSED_DIR / 'mens_odi_train.parquet')
womens_odi_test = pd.read_parquet(PROCESSED_DIR / 'womens_odi_test.parquet')

X_womens_test, y_womens_test, features = prepare_features(womens_odi_test)

# Test Men's ODI models on Women's ODI data
transfer_preds = {}

for model_name in ['xgboost', 'random_forest', 'lightgbm', 'neural_network']:
    try:
        model, scaler = load_model(model_name, format_key='mens_odi')
        preds = predict_with_model(model, X_womens_test, scaler=scaler)
        display_name = model_name.replace('_', ' ').title()
        transfer_preds[f"{display_name} (Transfer)"] = preds
        metrics = compute_metrics(y_womens_test.values, preds)
        print(f"{display_name} (Men's→Women's): RMSE={metrics['RMSE']:.2f}, R²={metrics['R2']:.4f}")
    except Exception as e:
        print(f"{model_name}: Failed - {e}")

# Compare with Women's ODI native models
print("\nNative Women's ODI results for comparison:")
print(womens_odi_results['eval_results']['overall'][['Model', 'RMSE', 'R2']].to_string(index=False))

In [ ]:
# Summary
print("\n" + "="*60)
print("CROSS-FORMAT ANALYSIS SUMMARY")
print("="*60)
for fmt_name, df in format_results.items():
    best_model = df.loc[df['RMSE'].idxmin()]
    dls_row = df[df['Model'] == 'DLS']
    if len(dls_row) > 0:
        dls_rmse = dls_row.iloc[0]['RMSE']
        improvement = ((dls_rmse - best_model['RMSE']) / dls_rmse) * 100
        print(f"\n{fmt_name}:")
        print(f"  Best model: {best_model['Model']} (RMSE={best_model['RMSE']:.2f})")
        print(f"  DLS baseline: RMSE={dls_rmse:.2f}")
        print(f"  Improvement: {improvement:.1f}%")

print("\n✓ Extension analysis complete!")